<a href="https://colab.research.google.com/github/elhamod/IS883_Fall_2026/blob/main/Week%2001/IS883_Week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Language Modeling with N-grams on a Real Book Corpus

### What you will do in this notebook

In class we built a tiny n-gram on three toy sentences. Here we scale it up to a **real corpus**: a few hundred thousand words of public-domain children's literature (*Alice in Wonderland*, *Stories to Tell to Children*, *The Adventures of Buster Bear*, and *The Parent's Assistant*), all from Project Gutenberg.

You will:

1. Load and inspect a large text corpus.
2. Train n-gram language models on it.
3. See what changes when you change **n**.
4. See that the same model gives **different text every run** (stochasticity).
5. Use the model for **text completion** from different prefixes.
6. Measure model quality with **perplexity**.

# Part 0: Setup

In [ ]:
# Install the one library this notebook needs.
!pip install nltk --quiet

Machine learning is generally stochastic, meaning you get different results for different runs. To control that, we "seed" the code. Enter your BUID below; it is used as the seed everywhere in this notebook.

In [ ]:
# Your BUID seeds every random step so your results are reproducible.
BUID = 123456  # enter ONLY the numerical part

In [ ]:
# Seed Python's and NumPy's random number generators so runs are repeatable.
import random
import numpy as np

random.seed(BUID)
np.random.seed(BUID)

# Part 1: Getting a Large Text Corpus

`nltk` ships with a sample of Project Gutenberg books. We download it once and then pick the four books that are children's literature.

Notice how little code this takes: `gutenberg.sents(book)` hands us the book **already split into sentences and already split into words**. No manual text cleaning needed.

In [ ]:
# Download NLTK's sample of Project Gutenberg books and list what's available.
import nltk

nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import gutenberg

print("Available books:", gutenberg.fileids())

In [ ]:
# Load four children's books and combine them into one list of sentences.
# gutenberg.sents() returns each sentence already split into words.
books = [
    'carroll-alice.txt',          # Alice's Adventures in Wonderland
    'bryant-stories.txt',         # Stories to Tell to Children
    'burgess-busterbrown.txt',    # The Adventures of Buster Bear
    'edgeworth-parents.txt',      # The Parent's Assistant
]

sentences = []
for book in books:
    # Lower-case every word so "The" and "the" count as the same word
    book_sentences = [[word.lower() for word in sentence] for sentence in gutenberg.sents(book)]
    sentences = sentences + book_sentences
    print(f"{book:30s} {len(book_sentences):6d} sentences, {len(gutenberg.words(book)):7d} words")

print()
print("TOTAL:", len(sentences), "sentences,", sum(len(s) for s in sentences), "words")

Let's look at a few sentences to see what the model will actually be trained on.

In [ ]:
# Peek at the first few sentences to see what the model will train on.
for sentence in sentences[:5]:
    print(' '.join(sentence))
    print('---')

### Train / test split

We hold out part of the data. The model **only** sees the training sentences. The test sentences are used later to measure how well the model generalizes.

In [ ]:
# Shuffle, then hold out 10% of the sentences as a test set the model never sees.
random.Random(BUID).shuffle(sentences)   # shuffle so the split mixes all four books

training_split = 0.9
split_point = int(training_split * len(sentences))
train_sentences = sentences[:split_point]
test_sentences = sentences[split_point:]

print("training sentences:", len(train_sentences))
print("test sentences    :", len(test_sentences))

# Part 2: Training an N-gram Language Model

`nltk` does the heavy lifting. `padded_everygram_pipeline` adds the `<s>` (start) and `</s>` (end) markers and counts every n-gram;

We wrap it in a small function so we can train a model for any `n` with one line.

In [ ]:
# Train an n-gram language model from a list of sentences.
# Lidstone adds a tiny amount to every word count so word combinations the model
# never saw still get a small (non-zero) probability instead of breaking the math.
from nltk.lm import Lidstone
from nltk.lm.preprocessing import padded_everygram_pipeline

def train_ngram_model(n, sentences):
    train_data, vocabulary = padded_everygram_pipeline(n, sentences)   # add <s>/</s> and count n-grams
    model = Lidstone(0.01, n)
    model.fit(train_data, vocabulary)
    return model

my_ngram_model = train_ngram_model(4, train_sentences)

print("n =", my_ngram_model.order)
print("vocabulary size:", len(my_ngram_model.vocab), "distinct words")

**Experiment.** Re-run the cell above with a different value of `n` (change the `4`). How does the vocabulary size change?

### What did the model actually learn?

An n-gram model is essentially a big lookup table: *given the previous word(s), how often did each word come next?* Let's look up a few phrases.

In [ ]:
# Helper: show the words most likely to follow a given phrase, with their probabilities.
def show_next_words(model, previous_words, how_many=8):
    if isinstance(previous_words, str):
        previous_words = previous_words.split()

    if len(previous_words) > model.order - 1:    # an n-gram model can only look back n-1 words
        print(f'This model (n = {model.order}) can only look back {model.order - 1} word(s), '
              f'but you gave it {len(previous_words)}. Train a model with n = {len(previous_words) + 1}.\n')
        return

    counts = model.counts[previous_words] if len(previous_words) > 0 else model.counts.unigrams
    total = sum(counts.values())
    phrase = " ".join(previous_words)

    if total == 0:
        print(f'"{phrase}" never appears in the training text.\n')
        return

    print(f'After "{phrase}" ({total} times in the training text), the most likely next words are:')
    for word, count in counts.most_common(how_many):
        print(f"   {word:12s} probability = {count / total:.3f}")
    print()

show_next_words(my_ngram_model, ["little"])
show_next_words(my_ngram_model, ["once", "upon"])
show_next_words(my_ngram_model, ["said"])

**Experiment.** Look up other words of your own by changing the words passed to `show_next_words` above. Can you find a word whose most-likely next words surprise you?

# Part 3: Generating Text

To generate text we repeatedly ask the model: *given the last n-1 words, what word comes next?* We stop when the model produces the end-of-sentence marker `</s>`.

The `seed` argument makes a run reproducible. If you leave it out, you get a different sample every time (we use that on purpose in Part 5).

In [ ]:
# Generate text one word at a time, stopping at the sentence-end marker.
# Pass a prefix to continue from given words; pass a seed to make a run repeatable.
def generate_text(model, num_words=40, prefix=None, seed=None):
    n = model.order
    random_generator = random.Random(seed) if seed is not None else None

    context = ["<s>"] * (n - 1)          # pretend we are at the start of a sentence
    generated = []

    if prefix is not None:               # start from words the user gave us
        prefix_words = nltk.word_tokenize(prefix.lower())
        context = context + prefix_words
        generated = list(prefix_words)

    for _ in range(num_words):
        last_words = context[-(n - 1):] if n > 1 else []
        next_word = model.generate(text_seed=last_words, random_seed=random_generator)
        if next_word == "</s>":          # the model decided the sentence is over
            break
        if next_word == "<s>":           # skip start markers
            continue
        generated.append(next_word)
        context.append(next_word)

    return " ".join(generated)


print(generate_text(my_ngram_model, num_words=40, prefix="once", seed=None))

**Experiment.** Set `seed=BUID` and re-run a few times — the text stays the same. Then change the `prefix` to a phrase of your own and see what the model writes.

# Part 4: Trying Different Values of n

`n` controls **how much context** the model is allowed to look at.

Below we train one model per value of `n` and generate a sentence from each, using the same seed so the comparison is fair.

**Warning:** training the larger models takes ~10 seconds each, so this cell takes a minute or two.

In [ ]:
# Train one model per value of n and print sample sentences from each.
# The same seed is used across models so the comparison is fair.
# Warning: the larger models take ~10s each, so this cell runs for a minute or two.
models = {}   # save the models so we can reuse them later

for n in [1, 2, 3, 4, 5]:
    models[n] = train_ngram_model(n, train_sentences)
    print(f"===== n = {n} =====")
    for sample in range(3):
        print(" ", generate_text(models[n], num_words=40, seed=BUID + 101 * n + sample))
    print()

**Question 1.** Describe how the generated text changes as `n` grows. Which value of `n` gives the best-sounding sentences? What starts to go wrong at the high end?

**Answer**

*Provide your answer here*

# Part 5: Stochasticity — Same Model, Different Runs

An n-gram model does not output *the* next word; it outputs a **probability distribution** over next words, and we sample from it. So the same model, asked the same question, answers differently each time.

Here we run the same trigram model five times **without** fixing a seed.

In [ ]:
# Run the SAME model five times with NO seed -> different text each run.
trigram_model = train_ngram_model(n=3, sentences=train_sentences)

for run in range(5):
    print(f"Run {run + 1}: {generate_text(trigram_model, num_words=30)}")
    print("---")

**Experiment.** Change `n` in the cell above (try 2 or 5) and re-run. Do the five runs differ from each other more, or less?

**Question 2.** Why do we get different text on every run? And why is being able to fix the seed useful?

**Answer**

*Provide your answer here*

# Part 6: Text Completion with Different Prefixes

So far we started from an empty sentence. But we can also hand the model a **prefix** and ask it to continue — which is exactly what auto-complete on your phone does, and a simplified version of what ChatGPT does.

In [ ]:
# Give the model a starting phrase (prefix) and let it finish the sentence.
demo_model = train_ngram_model(n=5, sentences=train_sentences)

prefixes = [
    "once upon a time",
    "the little girl",
    "alice was",
    "he said to",
    "the big data analyst",   # a prefix that does NOT belong to children's books
]

for i, prefix in enumerate(prefixes):
    print(f'PREFIX: "{prefix}"')
    print("  ->", generate_text(demo_model, num_words=30, prefix=prefix, seed=BUID + i))
    print("---")

The same prefix through models of different `n`:

In [ ]:
# Complete the same prefix with each n, and show the next-word options at each step.
prefix = "once upon a time"

for n in [1, 2, 3, 4, 5]:
    prefix_words = prefix.split(" ")
    last_words = prefix_words[-(n - 1):] if n > 1 else []   # the n-1 words the model looks back at
    print(f"n = {n}:")
    print("  ", generate_text(models[n], num_words=30, prefix=prefix, seed=BUID))
    show_next_words(models[n], last_words)
    print("---")

And the same prefix, same model, several times — stochasticity again, now in a completion setting:

In [ ]:
# Same prefix, same model, five times -> stochasticity again, now while completing text.
prefix = "the little girl"

for run in range(5):
    print(generate_text(demo_model, num_words=25, prefix=prefix, seed=BUID + run))
    print("---")

**Question 3.** What happened with the prefix `"the big data analyst"`? Why?

**Answer**

*Provide your answer here*

# Part 7: Measuring Quality with Perplexity

Generated text is fun but subjective. **Perplexity** is the standard numeric score for a language model.

Intuition: perplexity is roughly *"on average, how many words was the model surprised with at each step?"* A perplexity of 10 means the model was about as uncertain as if it had been choosing uniformly among 10 words. **Lower is better.**

We compute it on text the model has never seen, since scoring the model on its own training text would be like grading a student on the exact questions they memorized.

In [ ]:
# Train a bigram model to demonstrate perplexity.
bigram_model = train_ngram_model(n=2, sentences=train_sentences)

In [ ]:
# Compute the perplexity of a single sentence (lower = the model is less surprised).
from nltk.util import ngrams
from nltk.lm.preprocessing import pad_both_ends

def sentence_perplexity(model, sentence):
    n = model.order
    tokens = nltk.word_tokenize(sentence.lower())
    tokens = list(pad_both_ends(tokens, n))       # add <s> ... </s>
    tokens = model.vocab.lookup(tokens)           # replace unknown words with <UNK>
    return model.perplexity(list(ngrams(tokens, n)))


test_sentence = "once upon a time there was a little boy."
print("perplexity =", sentence_perplexity(bigram_model, test_sentence))

### Comparing sentences

Now that perplexity is finite, we can use it to ask: *how surprised is a children's-book model by this sentence?*

In [ ]:
# Score several sentences: children's-book style should be far less perplexing than business jargon.
sentences_to_score = [
    "once upon a time there was a little boy.",
    "the little girl said to her mother.",
    "alice was beginning to get very tired.",
    "the quarterly earnings report exceeded analyst expectations.",
    "blockchain optimizes shareholder value through synergistic frameworks.",
]

for sentence in sentences_to_score:
    score = sentence_perplexity(bigram_model, sentence)
    print(f"{score:10.1f}   {sentence}")

### Perplexity vs. n

We measure perplexity on the **training** sentences and on the **held-out test** sentences, for each value of `n`.

**Warning:** this cell trains five models and scores them, so it takes a couple of minutes.

In [ ]:
# Measure perplexity on the training and the held-out test sentences, for each n.
# Warning: this trains and scores five models, so it takes a couple of minutes.
def corpus_perplexity(model, sentences):
    n = model.order
    all_ngrams = []
    for sentence in sentences:
        tokens = model.vocab.lookup(list(pad_both_ends(sentence, n)))
        all_ngrams = all_ngrams + list(ngrams(tokens, n))
    return model.perplexity(all_ngrams)


range_of_n = [1, 2, 3, 4, 5]
train_perplexities = []
test_perplexities = []

for n in range_of_n:
    model = train_ngram_model(n, train_sentences)
    train_perplexities.append(corpus_perplexity(model, train_sentences))
    test_perplexities.append(corpus_perplexity(model, test_sentences))
    print(f"n = {n}:  train perplexity = {train_perplexities[-1]:8.1f}   test perplexity = {test_perplexities[-1]:8.1f}")

In [ ]:
# Plot training vs. test perplexity against n. The gap between the lines is overfitting.
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(range_of_n, train_perplexities, label='Training Perplexity', color='blue', marker='o')
plt.plot(range_of_n, test_perplexities, label='Test Perplexity', color='red', marker='o')
plt.xlabel('n (size of the context window)')
plt.ylabel('Perplexity (lower is better)')
plt.title('Training and Test Perplexity vs. n')
plt.xticks(range_of_n)
plt.legend()
plt.grid(True)
plt.show()

**Question 4.** Read the plot. Which values of `n` underfit, which overfit, and which is best? Compare this plot to the accuracy-vs-depth plot from Week 1.

**Answer**

*Provide your answer here*

# Part 8: Follow up Questions and Exercises

**Exercise 1.** Retrain the models on a *different* genre from the same collection — for example `['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt']` (Jane Austen) or `['shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'shakespeare-caesar.txt']`. Generate text from the new trigram. Can you tell the genre from the generated text alone?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 2.** Score the sentence `"once upon a time there was a little boy."` with **both** the children's-book model and your new model from Exercise 1. Which model is less perplexed, and why? What does this tell you about what perplexity actually measures?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 3.** Rerun the notebook with less training data and more test data. How do the results change?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*